In [ ]:
from pyspark.sql import SparkSession
from pathlib import Path
from pyspark.sql import functions as F
import country_converter as coco
from itertools import chain

CLEANED_DATA_DIR = Path("../data/cleaned")
PROCESSED_DATA_DIR = Path("../data")

spark = SparkSession.builder.appName("profiling").config("spark.sql.ansi.enabled", "false").config("spark.driver.memory", "16g").getOrCreate()
df = spark.read.parquet(f"{CLEANED_DATA_DIR}/4c_eea_co2_emissions_from_passenger_cars-001.parquet")

df.show(10)

In [ ]:
import torch
import numpy as np
import pandas as pd
import pyspark.sql.functions as F
from VehicleAutoencoder import VehicleAutoencoder

def generate_latent_dataframe(
    spark_df, 
    checkpoint_path="best_vehicle_autoencoder.ckpt", 
    scaler_path="scaler_params.pt",
    device="cuda",
    batch_size=65536
) -> pd.DataFrame:
    device_obj = torch.device(device if torch.cuda.is_available() else "cpu")
    model = VehicleAutoencoder.load_from_checkpoint(checkpoint_path).to(device_obj)
    model.eval()

    scaler_params = torch.load(scaler_path, weights_only=False)
    feature_cols = scaler_params["feature_names"]

    motor_col = F.col("Motor energy")
    is_electric = motor_col == "Electricity"
    is_ice = motor_col.isin("Petrol (excluding hybrids)", "Diesel (excluding hybrids)")

    df_zeroed = spark_df.withColumn(
        "co2_emissions_WLTP (g/km)",
        F.when(is_electric, 0.0).otherwise(F.col("co2_emissions_WLTP (g/km)"))
    ).withColumn(
        "engine_capacity (cm3)",
        F.when(is_electric, 0.0).otherwise(F.col("engine_capacity (cm3)"))
    ).withColumn(
        "electric_energy_consumption (Wh/km)",
        F.when(is_ice, 0.0).otherwise(F.col("electric_energy_consumption (Wh/km)"))
    )

    pdf = df_zeroed.dropna(subset=feature_cols).toPandas()
    X_raw = pdf[feature_cols].values.astype(np.float32)
    
    X_scaled = (X_raw - scaler_params["mean"]) / scaler_params["std"]

    latent_coords = []
    with torch.no_grad():
        for i in range(0, len(X_scaled), batch_size):
            batch_x = torch.tensor(X_scaled[i : i + batch_size], dtype=torch.float32, device=device_obj)
            latent_coords.append(model.encode(batch_x).cpu().numpy())

    latent_matrix = np.vstack(latent_coords)
    pdf["z_1"] = latent_matrix[:, 0]
    pdf["z_2"] = latent_matrix[:, 1]

    return pdf

full_enriched_df = generate_latent_dataframe(df)

In [ ]:
full_enriched_df

In [ ]:
import numpy as np
import pandas as pd

def compute_grid_normalisation(
    df: pd.DataFrame, 
    grid_size: float = 0.2, 
    min_registrations: int = 10
) -> pd.DataFrame:
    """
    Computes latent space volume (normalisation factor) and normalized registrations.
    
    Parameters:
    -----------
    df : pd.DataFrame containing ['TIME_PERIOD', 'Motor energy', 'registrations', 'z_1', 'z_2']
    grid_size : Size of each grid cell square in latent z-score units (default: 0.2)
    min_registrations : Threshold to consider a grid cell active (filters out prototype noise)
    
    Returns:
    --------
    pd.DataFrame with normalisation factors and final normalized metrics per group.
    """
    pdf = df.copy()

    # continuous latent space into discrete cell coordinates
    pdf["cell_x"] = np.floor(pdf["z_1"] / grid_size).astype(int)
    pdf["cell_y"] = np.floor(pdf["z_2"] / grid_size).astype(int)

    # aggregate registrations at the grid cell level
    cell_agg = (
        pdf.groupby(["TIME_PERIOD", "Motor energy", "cell_x", "cell_y"])
        .agg(
            cell_registrations=("registrations", "sum"),
            unique_variants=("variant", "nunique") if "variant" in pdf.columns else ("registrations", "count")
        )
        .reset_index()
    )

    # filter noise
    active_cells = cell_agg[cell_agg["cell_registrations"] >= min_registrations]

    # compute the normalisation factor (latent volume = count of occupied active cells)
    volume_df = (
        active_cells.groupby(["TIME_PERIOD", "Motor energy"])
        .agg(
            latent_volume=("cell_x", "count"),
            active_registrations=("cell_registrations", "sum")
        )
        .reset_index()
    )

    # compute total raw registrations per powertrain group
    raw_totals = (
        pdf.groupby(["TIME_PERIOD", "Motor energy"])["registrations"]
        .sum()
        .reset_index()
        .rename(columns={"registrations": "total_raw_registrations"})
    )

    # merge and calculate normalized metric
    summary = pd.merge(volume_df, raw_totals, on=["TIME_PERIOD", "Motor energy"])
    summary["normalized_registrations"] = (
        summary["total_raw_registrations"] / summary["latent_volume"]
    )

    return summary

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

norm_summary = compute_grid_normalisation(
    full_enriched_df, 
    grid_size=0.2,
    min_registrations=10
)

baseline_summary = (
    full_enriched_df.groupby(["TIME_PERIOD", "Motor energy"])
    .agg(
        total_raw_registrations=("registrations", "sum"),
        unique_model_badges=("commercial_name", "nunique")
    )
    .reset_index()
)
baseline_summary["baseline_normalized_registrations"] = (
    baseline_summary["total_raw_registrations"] / baseline_summary["unique_model_badges"]
)

sns.set_theme(style="whitegrid", font_scale=1.1)
fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharex=True)

sns.lineplot(
    data=norm_summary,
    x="TIME_PERIOD",
    y="total_raw_registrations",
    hue="Motor energy",
    marker="o",
    linewidth=2.5,
    ax=axes[0]
)
axes[0].set_title("A: Unadjusted Raw Registrations", fontweight="bold", fontsize=13)
axes[0].set_xlabel("Year", fontsize=12)
axes[0].set_ylabel("Total Volume (Units)", fontsize=12)
axes[0].ticklabel_format(style="plain", axis="y")
axes[0].grid(True, linestyle="--", alpha=0.6)

sns.lineplot(
    data=baseline_summary,
    x="TIME_PERIOD",
    y="baseline_normalized_registrations",
    hue="Motor energy",
    marker="^",
    linewidth=2.5,
    ax=axes[1]
)
axes[1].set_title("B: Naive Baseline (Sales / Commercial Name)", fontweight="bold", fontsize=13)
axes[1].set_xlabel("Year", fontsize=12)
axes[1].set_ylabel("Registrations / Unique Model Name", fontsize=12)
axes[1].grid(True, linestyle="--", alpha=0.6)

sns.lineplot(
    data=norm_summary,
    x="TIME_PERIOD",
    y="normalized_registrations",
    hue="Motor energy",
    marker="s",
    linewidth=2.5,
    ax=axes[2]
)
axes[2].set_title("C: Latent Grid Density (Sales / Latent Cell)", fontweight="bold", fontsize=13)
axes[2].set_xlabel("Year", fontsize=12)
axes[2].set_ylabel("Density per Physical Unit Space", fontsize=12)
axes[2].grid(True, linestyle="--", alpha=0.6)

plt.suptitle(
    "EEA Fleet Analysis: Raw Registration Volume vs. Naive Baseline vs. Latent Variety Normalization",
    y=1.03, 
    fontweight="bold", 
    fontsize=15
)
plt.tight_layout()
plt.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

norm_summary = norm_summary.sort_values(["Motor energy", "TIME_PERIOD"])
baseline_summary = baseline_summary.sort_values(["Motor energy", "TIME_PERIOD"])

all_powertrains = sorted(list(
    set(norm_summary["Motor energy"].unique()).union(set(baseline_summary["Motor energy"].unique()))
))
colors = px.colors.qualitative.Plotly
color_map = {ptype: colors[i % len(colors)] for i, ptype in enumerate(all_powertrains)}

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        "A: Unadjusted Raw Registrations",
        "B: Naive Baseline (Sales / Commercial Name)",
        "C: Latent Grid Density (Sales / Latent Cell)"
    ),
    shared_xaxes=True
)

for ptype in all_powertrains:
    df_norm = norm_summary[norm_summary["Motor energy"] == ptype]
    df_base = baseline_summary[baseline_summary["Motor energy"] == ptype]
    
    if not df_norm.empty:
        fig.add_trace(
            go.Scatter(
                x=df_norm["TIME_PERIOD"],
                y=df_norm["total_raw_registrations"],
                mode="lines+markers",
                name=ptype,
                legendgroup=ptype,
                showlegend=True,
                line=dict(color=color_map[ptype], width=2.5),
                marker=dict(symbol="circle", size=8),
                connectgaps=True
            ),
            row=1, col=1
        )
    
    if not df_base.empty:
        fig.add_trace(
            go.Scatter(
                x=df_base["TIME_PERIOD"],
                y=df_base["baseline_normalized_registrations"],
                mode="lines+markers",
                name=ptype,
                legendgroup=ptype,
                showlegend=False,
                line=dict(color=color_map[ptype], width=2.5),
                marker=dict(symbol="triangle-up", size=8),
                connectgaps=True
            ),
            row=1, col=2
        )
    
    if not df_norm.empty:
        fig.add_trace(
            go.Scatter(
                x=df_norm["TIME_PERIOD"],
                y=df_norm["normalized_registrations"],
                mode="lines+markers",
                name=ptype,
                legendgroup=ptype,
                showlegend=False,
                line=dict(color=color_map[ptype], width=2.5),
                marker=dict(symbol="square", size=8),
                connectgaps=True
            ),
            row=1, col=3
        )

fig.update_xaxes(title_text="Year", gridcolor="rgba(0,0,0,0.1)")
fig.update_yaxes(title_text="Total Volume (Units)", row=1, col=1, gridcolor="rgba(0,0,0,0.1)")
fig.update_yaxes(title_text="Registrations / Commercial Name", row=1, col=2, gridcolor="rgba(0,0,0,0.1)")
fig.update_yaxes(title_text="Density per Physical Unit Space", row=1, col=3, gridcolor="rgba(0,0,0,0.1)")

fig.update_layout(
    title_text="<b>EEA Fleet Analysis: Raw Volume vs. Naive Baseline vs. Latent Normalization</b>",
    template="plotly_white",
    height=500,
    width=1350,
    legend_title_text="Motor energy",
    hovermode="x unified"
)

fig.show()